# Week 11 Companion Notebook: Pipeline Orchestration & Data Quality

## ISM 6562 — Big Data for Business Applications

---

This notebook accompanies the Week 11 lecture on **Apache Airflow for pipeline orchestration and data quality engineering**. After building storage (Week 8), transformation (Week 9), and streaming (Week 10) layers, this week ties them together with **scheduled, observable, production-grade pipelines**.

You will work through two complete business scenarios:

- **HealthPulse (Lambda Architecture)** — Orchestrating the daily batch pipeline with quality gates, backfill, and idempotency
- **GreenRoute (Kappa Architecture)** — Weekly fleet reporting with GPS data quality validation via Spark

By the end of this notebook, you will have hands-on experience with:

1. Navigating and operating the Airflow Web UI
2. Understanding DAG structure: operators, dependencies, schedules, retries
3. Triggering DAGs manually and monitoring task execution via CLI
4. Building data quality checks (null rates, value ranges, coordinate validation)
5. Implementing quality gates that halt downstream tasks on failure
6. Running backfill operations and verifying idempotency
7. Mapping the on-prem stack to AWS managed services

### Important: This notebook runs on your local machine

Unlike previous weeks, there is no Jupyter container inside Docker. Airflow is accessed through its **Web UI** (http://localhost:8080) and **CLI** (via `docker exec`). This notebook runs on your host machine and uses shell commands to interact with the containers.

---

## Part 0: Setup & Environment Verification

The `docker-compose.yml` for this week creates an orchestration environment with the following components:

| Component | Container Name | Role |
|---|---|---|
| **PostgreSQL** | `airflow-postgres` | Airflow metadata database (DAG state, task history, XCom) |
| **Airflow Init** | `airflow-init` | One-shot container that initializes DB schema and creates admin user |
| **Airflow Webserver** | `airflow-webserver` | Web UI at port 8080 (DAG visualization, logs, manual triggers) |
| **Airflow Scheduler** | `airflow-scheduler` | Scans DAG files, evaluates schedules, queues and executes tasks |
| **Spark Master** | `spark-master` | Coordinates Spark cluster (UI on port 8081) |
| **Spark Worker** | `spark-worker` | Executes Spark jobs (2 cores, 2 GB RAM) |

**Architecture overview:**

```
Airflow Scheduler → triggers DAG tasks → BashOperator / PythonOperator
                                        → spark-submit to Spark Master
Shared /data volume lets Airflow and Spark read/write the same files.
```

In [ ]:
%%bash
# Start the Week 11 Docker cluster
# Run this from the week11 directory
cd week11-pipeline-orchestration-modern-data-stack
docker compose up -d
echo ""
echo "Containers starting. Airflow needs ~60 seconds to initialize."
echo "The airflow-init container will exit after setup — that is expected."

In [ ]:
# Wait for Airflow to fully initialize before proceeding
import time

print("Waiting 60 seconds for Airflow to initialize...")
print("(The scheduler needs time to parse DAG files and the webserver needs to start.)")
time.sleep(60)
print("Done waiting. Let's verify the environment.")

In [ ]:
%%bash
# Verify Airflow is running
echo "=== Airflow Version ==="
docker exec airflow-webserver airflow version

echo ""
echo "=== Container Status ==="
docker ps --format "table {{.Names}}\t{{.Status}}\t{{.Ports}}" | grep -E "airflow|spark|postgres"

echo ""
echo "=== DAGs Loaded ==="
docker exec airflow-webserver airflow dags list

echo ""
echo "If you see both DAGs listed above, Airflow is ready."
echo "Open the Web UI at: http://localhost:8080"
echo "Login: admin / admin"

In [ ]:
# Download Week 11 data files from the course data repository
import os
import urllib.request

REPO = "https://raw.githubusercontent.com/prof-tcsmith/ism6562s26-class/main"
DATA_DIR = "week11-pipeline-orchestration-modern-data-stack/data"

files_to_download = [
    # HealthPulse daily patient feeds
    ("week11/data/healthpulse/daily-patient-feed/2026-04-28.csv", "healthpulse/daily-patient-feed/2026-04-28.csv"),
    ("week11/data/healthpulse/daily-patient-feed/2026-04-29.csv", "healthpulse/daily-patient-feed/2026-04-29.csv"),
    ("week11/data/healthpulse/daily-patient-feed/2026-04-30.csv", "healthpulse/daily-patient-feed/2026-04-30.csv"),
    ("week11/data/healthpulse/quality-test-bad-data.csv", "healthpulse/quality-test-bad-data.csv"),
    # GreenRoute weekly GPS summaries
    ("week11/data/greenroute/weekly-gps-summary/week-2026-W16.csv", "greenroute/weekly-gps-summary/week-2026-W16.csv"),
    ("week11/data/greenroute/weekly-gps-summary/week-2026-W17.csv", "greenroute/weekly-gps-summary/week-2026-W17.csv"),
    ("week11/data/greenroute/quality-test-bad-gps.json", "greenroute/quality-test-bad-gps.json"),
]

downloaded = 0
for remote_path, local_path in files_to_download:
    full_local = os.path.join(DATA_DIR, local_path)
    os.makedirs(os.path.dirname(full_local), exist_ok=True)
    url = f"{REPO}/{remote_path}"
    try:
        urllib.request.urlretrieve(url, full_local)
        size = os.path.getsize(full_local)
        print(f"  Downloaded: {local_path} ({size:,} bytes)")
        downloaded += 1
    except Exception as e:
        print(f"  FAILED: {local_path} — {e}")

print(f"\nDownloaded {downloaded}/{len(files_to_download)} files to {DATA_DIR}/")
print("These files are mounted into Airflow containers at /opt/airflow/data/")
print("and into Spark containers at /data/")

---

## Part 1: Airflow Fundamentals

Apache Airflow is the industry-standard open-source tool for **orchestrating data pipelines**. It provides:

- **DAGs** (Directed Acyclic Graphs) to define task dependencies
- **Scheduling** to run pipelines on cron-like schedules
- **Monitoring** via a rich web UI with logs, task states, and run history
- **Retry logic** for resilience against transient failures
- **XCom** for passing data between tasks

### 1a. Exploring the Web UI

Open **http://localhost:8080** in your browser and log in with **admin / admin**.

You should see two pre-loaded DAGs:

| DAG ID | Schedule | Description |
|---|---|---|
| `healthpulse_daily_pipeline` | `0 2 * * *` (daily at 2 AM) | Patient data ingestion, validation, transformation, loading |
| `greenroute_weekly_report` | `0 2 * * MON` (Mondays at 2 AM) | GPS validation, Spark aggregation, fleet reporting |

**Explore these views** (click on a DAG name to enter its detail page):

1. **Grid view** — Shows run history as a grid of task states over time
2. **Graph view** — Shows the DAG structure with nodes (tasks) and edges (dependencies)
3. **Code** — Shows the actual Python DAG file (read-only)

Take a moment to familiarize yourself with the UI before proceeding.

### 1b. DAG Anatomy — The HealthPulse Pipeline

Let's read the actual DAG file that Airflow is executing. This file lives in the `dags/` directory, which is mounted into both the webserver and scheduler containers.

In [ ]:
%%bash
# Display the full HealthPulse DAG file
echo "=== dags/healthpulse_daily_pipeline.py ==="
echo ""
docker exec airflow-webserver cat /opt/airflow/dags/healthpulse_daily_pipeline.py

**Key components explained:**

| Component | What It Does |
|---|---|
| `default_args` | Applied to every task — sets retries (2), retry delay (5 min), timeout (30 min) |
| `DAG(...)` | Defines the DAG: ID, schedule (`0 2 * * *`), start date, and tags |
| `BashOperator` | Runs shell commands (used for ingest, transform, load steps) |
| `PythonOperator` | Runs Python functions (used for validation, quality report, notification) |
| `{{ ds }}` | Jinja template — Airflow replaces this with the execution date (e.g., `2026-04-28`) |
| `xcom_push/pull` | XCom (cross-communication) — passes values between tasks |
| `>> operator` | Defines task dependencies: `A >> B` means "B runs after A" |

**Pipeline structure (diamond dependency pattern):**

```
  ingest_patient_feed
         |
    validate_data          <-- Quality Gate: fails if error rate > 5%
       /    \
  transform   quality_report   <-- These run IN PARALLEL after validation passes
       \    /
    load_to_warehouse
         |
    send_notification
```

Notice the **diamond pattern**: after validation, `transform_data` and `quality_report` can run simultaneously because they are independent. Both must complete before `load_to_warehouse` starts. This is a common pattern for maximizing parallelism while maintaining correctness.

### 1c. Triggering a DAG

DAGs can be triggered in two ways:

1. **Automatically** — by the scheduler based on the cron schedule
2. **Manually** — from the Web UI (play button) or via the CLI

Let's trigger the HealthPulse DAG manually using the CLI.

In [ ]:
%%bash
# First, unpause the DAG (newly loaded DAGs are paused by default)
docker exec airflow-webserver airflow dags unpause healthpulse_daily_pipeline
echo ""

# Trigger a manual run
docker exec airflow-webserver airflow dags trigger healthpulse_daily_pipeline
echo ""
echo "DAG triggered! Check the Web UI at http://localhost:8080 to see it running."

In [ ]:
# Wait a moment for the run to complete, then check status
import time
print("Waiting 30 seconds for the DAG run to complete...")
time.sleep(30)

In [ ]:
%%bash
# Check the status of DAG runs
echo "=== HealthPulse DAG Runs ==="
docker exec airflow-webserver airflow dags list-runs -d healthpulse_daily_pipeline

echo ""
echo "=== Task Instance States ==="
docker exec airflow-webserver airflow tasks states-for-dag-run healthpulse_daily_pipeline $(docker exec airflow-webserver airflow dags list-runs -d healthpulse_daily_pipeline -o plain --no-header | head -1 | awk '{print $3}') 2>/dev/null || echo "(Run the trigger cell first if no runs appear)"

### 1d. Task Dependencies

Airflow tracks which tasks depend on which. Let's visualize the dependency tree from the CLI.

In [ ]:
%%bash
# Show the task dependency tree for the HealthPulse DAG
echo "=== HealthPulse Task Tree ==="
docker exec airflow-webserver airflow tasks list healthpulse_daily_pipeline --tree

echo ""
echo "=== GreenRoute Task Tree ==="
docker exec airflow-webserver airflow tasks list greenroute_weekly_report --tree

**Reading the dependency tree:**

- Indentation shows parent-child relationships
- Tasks at the same indentation level under the same parent can run **in parallel**
- A task only starts when **all** its upstream (parent) tasks have succeeded

In the HealthPulse DAG:

- `ingest_patient_feed` runs first (no upstream dependencies)
- `validate_data` waits for ingestion to complete
- `transform_data` and `quality_report` both wait for validation — and can run at the same time
- `load_to_warehouse` waits for **both** transform and quality report to finish
- `send_notification` runs last

### 1e. Cron Expressions

Airflow uses standard **cron expressions** to define schedules. The format is:

```
┌───────────── minute (0-59)
│ ┌───────────── hour (0-23)
│ │ ┌───────────── day of month (1-31)
│ │ │ ┌───────────── month (1-12)
│ │ │ │ ┌───────────── day of week (0-7, SUN-SAT)
│ │ │ │ │
* * * * *
```

**Exercise:** Match each business requirement to its cron expression.

| Business Requirement | Cron Expression |
|---|---|
| Every day at 2:00 AM | `0 2 * * *` |
| Every Monday at midnight | `0 0 * * 1` |
| Every 6 hours | `0 */6 * * *` |
| First day of each month at 3 AM | `0 3 1 * *` |
| Every weekday at 8 AM | `0 8 * * 1-5` |
| Every 15 minutes | `*/15 * * * *` |

The HealthPulse DAG uses `0 2 * * *` — daily at 2 AM. Why 2 AM? Because:

1. Upstream systems (EHR exports) finish by midnight
2. Results are ready before the morning clinical team shift at 7 AM
3. Infrastructure load is low during off-peak hours

In [ ]:
# Interactive exercise: Verify your cron understanding
# The croniter library can parse cron expressions and show upcoming runs

from datetime import datetime

# Our two DAG schedules
schedules = {
    "healthpulse_daily_pipeline": "0 2 * * *",
    "greenroute_weekly_report": "0 2 * * MON",
}

print("DAG Schedule Summary:")
print("=" * 60)
for dag_id, cron_expr in schedules.items():
    print(f"\n  DAG: {dag_id}")
    print(f"  Cron: {cron_expr}")
    if cron_expr == "0 2 * * *":
        print(f"  Meaning: Every day at 2:00 AM")
        print(f"  Next 3 runs: 2026-04-29 02:00, 2026-04-30 02:00, 2026-05-01 02:00")
    elif cron_expr == "0 2 * * MON":
        print(f"  Meaning: Every Monday at 2:00 AM")
        print(f"  Next 3 runs: 2026-05-04 02:00, 2026-05-11 02:00, 2026-05-18 02:00")

---

## Part 2: Data Quality Engineering

Data quality is the **most underappreciated aspect of data engineering**. A pipeline that runs on schedule but loads garbage data is worse than a pipeline that fails loudly — at least a failure gets noticed.

In this section, we build **data quality checks** that serve as gates in our Airflow pipelines.

### 2.1 The Bad Data Problem

Let's load a deliberately dirty dataset and see what problems lurk inside.

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

# Load the intentionally dirty HealthPulse data
bad_data = pd.read_csv("week11-pipeline-orchestration-modern-data-stack/data/healthpulse/quality-test-bad-data.csv")

print(f"Dataset shape: {bad_data.shape}")
print(f"\nColumn names: {list(bad_data.columns)}")
print(f"\nFirst 10 rows:")
bad_data.head(10)

In [ ]:
# Diagnose the data quality problems
print("=" * 60)
print("DATA QUALITY DIAGNOSIS")
print("=" * 60)

# Problem 1: Null patient IDs
null_patient_ids = bad_data["patient_id"].isna().sum()
print(f"\n1. Null patient_id values: {null_patient_ids} ({null_patient_ids/len(bad_data)*100:.1f}%)")
print(f"   Impact: Cannot link records to patients. These records are useless.")

# Problem 2: Negative costs
if "cost" in bad_data.columns:
    negative_costs = (bad_data["cost"] < 0).sum()
    print(f"\n2. Negative cost values: {negative_costs} ({negative_costs/len(bad_data)*100:.1f}%)")
    print(f"   Impact: Distorts financial reporting. Could indicate refunds or data entry errors.")
    if negative_costs > 0:
        print(f"   Examples: {bad_data[bad_data['cost'] < 0]['cost'].head(5).tolist()}")

# Problem 3: Future dates
if "admission_date" in bad_data.columns:
    bad_data["admission_date_parsed"] = pd.to_datetime(bad_data["admission_date"], errors="coerce")
    today = pd.Timestamp("2026-04-28")  # Our reference date
    future_dates = (bad_data["admission_date_parsed"] > today).sum()
    print(f"\n3. Future admission dates: {future_dates} ({future_dates/len(bad_data)*100:.1f}%)")
    print(f"   Impact: Impossible data — patients cannot be admitted in the future.")

# Problem 4: General null analysis
print(f"\n4. Null values per column:")
null_counts = bad_data.isnull().sum()
for col, count in null_counts.items():
    if count > 0:
        print(f"   {col}: {count} nulls ({count/len(bad_data)*100:.1f}%)")

total_issues = null_patient_ids
if "cost" in bad_data.columns:
    total_issues += negative_costs
if "admission_date" in bad_data.columns:
    total_issues += future_dates

print(f"\n" + "=" * 60)
print(f"TOTAL ISSUES FOUND: {total_issues} across {len(bad_data)} records")
print(f"Would this data pass a 5% error threshold? {'NO' if total_issues/len(bad_data) > 0.05 else 'YES'}")
print("=" * 60)

### 2.2 Building Quality Checks

Now let's build **reusable quality check functions** that can be plugged into any Airflow DAG. Each function returns a result dict and raises an exception if the check fails.

In [ ]:
def check_null_rate(df, column, threshold=0.05):
    """
    Check that the null rate for a column does not exceed the threshold.
    
    Args:
        df: pandas DataFrame
        column: column name to check
        threshold: maximum acceptable null rate (0.05 = 5%)
    
    Returns:
        dict with check results
    
    Raises:
        ValueError if null rate exceeds threshold
    """
    null_count = df[column].isna().sum()
    null_rate = null_count / len(df)
    passed = null_rate <= threshold
    
    result = {
        "check": "null_rate",
        "column": column,
        "null_count": null_count,
        "null_rate": round(null_rate, 4),
        "threshold": threshold,
        "passed": passed,
    }
    
    status = "PASSED" if passed else "FAILED"
    print(f"  [{status}] null_rate({column}): {null_rate:.2%} (threshold: {threshold:.0%})")
    
    if not passed:
        raise ValueError(
            f"Null rate check FAILED for '{column}': "
            f"{null_rate:.2%} exceeds {threshold:.0%} threshold"
        )
    return result


def check_value_range(df, column, min_val=None, max_val=None):
    """
    Check that all non-null values in a column fall within [min_val, max_val].
    
    Args:
        df: pandas DataFrame
        column: column name to check
        min_val: minimum acceptable value (inclusive), or None to skip
        max_val: maximum acceptable value (inclusive), or None to skip
    
    Returns:
        dict with check results
    
    Raises:
        ValueError if any values are out of range
    """
    series = df[column].dropna()
    violations = 0
    
    if min_val is not None:
        violations += (series < min_val).sum()
    if max_val is not None:
        violations += (series > max_val).sum()
    
    violation_rate = violations / len(df)
    passed = violations == 0
    
    result = {
        "check": "value_range",
        "column": column,
        "min_val": min_val,
        "max_val": max_val,
        "violations": int(violations),
        "violation_rate": round(violation_rate, 4),
        "passed": passed,
    }
    
    range_str = f"[{min_val}, {max_val}]"
    status = "PASSED" if passed else "FAILED"
    print(f"  [{status}] value_range({column}): {violations} violations outside {range_str}")
    
    if not passed:
        raise ValueError(
            f"Value range check FAILED for '{column}': "
            f"{violations} values outside {range_str}"
        )
    return result


def check_date_freshness(df, column, max_days=7, reference_date=None):
    """
    Check that dates in a column are not older than max_days from the reference date,
    and that no dates are in the future.
    
    Args:
        df: pandas DataFrame
        column: column name containing dates
        max_days: maximum age in days
        reference_date: the "current" date to check against (default: 2026-04-28)
    
    Returns:
        dict with check results
    
    Raises:
        ValueError if data is stale or contains future dates
    """
    if reference_date is None:
        reference_date = pd.Timestamp("2026-04-28")
    
    dates = pd.to_datetime(df[column], errors="coerce")
    valid_dates = dates.dropna()
    
    future_count = (valid_dates > reference_date).sum()
    cutoff = reference_date - pd.Timedelta(days=max_days)
    stale_count = (valid_dates < cutoff).sum()
    
    total_violations = future_count + stale_count
    passed = total_violations == 0
    
    result = {
        "check": "date_freshness",
        "column": column,
        "future_dates": int(future_count),
        "stale_dates": int(stale_count),
        "max_days": max_days,
        "passed": passed,
    }
    
    status = "PASSED" if passed else "FAILED"
    print(f"  [{status}] date_freshness({column}): {future_count} future, {stale_count} stale (>{max_days} days)")
    
    if not passed:
        raise ValueError(
            f"Date freshness check FAILED for '{column}': "
            f"{future_count} future dates, {stale_count} stale dates"
        )
    return result


print("Quality check functions defined: check_null_rate, check_value_range, check_date_freshness")

In [ ]:
# Run the quality checks against the bad data
print("Running quality checks against quality-test-bad-data.csv")
print("=" * 60)

results = []
checks = [
    ("check_null_rate", lambda: check_null_rate(bad_data, "patient_id", threshold=0.05)),
    ("check_value_range (cost >= 0)", lambda: check_value_range(bad_data, "cost", min_val=0) if "cost" in bad_data.columns else {"skipped": True}),
    ("check_date_freshness", lambda: check_date_freshness(bad_data, "admission_date", max_days=30) if "admission_date" in bad_data.columns else {"skipped": True}),
]

for check_name, check_fn in checks:
    try:
        result = check_fn()
        results.append((check_name, "PASSED", result))
    except ValueError as e:
        results.append((check_name, "FAILED", str(e)))
    except Exception as e:
        results.append((check_name, "ERROR", str(e)))

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
for name, status, detail in results:
    print(f"  {status}: {name}")

failed = sum(1 for _, s, _ in results if s == "FAILED")
print(f"\n{failed} of {len(results)} checks FAILED.")
if failed > 0:
    print("In an Airflow pipeline, this would HALT all downstream tasks.")

### 2.3 GPS Data Quality

GPS data has its own unique quality challenges. Bad coordinates can cause vehicles to appear in the middle of the ocean or move at impossible speeds.

In [ ]:
import json

# Load the intentionally bad GPS data
with open("week11-pipeline-orchestration-modern-data-stack/data/greenroute/quality-test-bad-gps.json") as f:
    bad_gps_raw = json.load(f)

bad_gps = pd.DataFrame(bad_gps_raw)
print(f"GPS dataset shape: {bad_gps.shape}")
print(f"Columns: {list(bad_gps.columns)}")
print(f"\nFirst 10 rows:")
bad_gps.head(10)

In [ ]:
# GPS-specific quality checks for US continental operations
print("GPS Data Quality Analysis")
print("=" * 60)

# Check 1: Latitude range for continental US (24 to 50 degrees)
lat_out_of_range = ((bad_gps["lat"] < 24) | (bad_gps["lat"] > 50)).sum()
print(f"\n1. Latitude outside US range [24, 50]: {lat_out_of_range} ({lat_out_of_range/len(bad_gps)*100:.1f}%)")

# Check 2: Longitude range for continental US (-130 to -65 degrees)
lon_out_of_range = ((bad_gps["lon"] < -130) | (bad_gps["lon"] > -65)).sum()
print(f"2. Longitude outside US range [-130, -65]: {lon_out_of_range} ({lon_out_of_range/len(bad_gps)*100:.1f}%)")

# Check 3: Null Island — coordinates of (0, 0)
# This is a common GPS default when no signal is received
null_island = ((bad_gps["lat"] == 0) & (bad_gps["lon"] == 0)).sum()
print(f"3. Null Island readings (0, 0): {null_island} ({null_island/len(bad_gps)*100:.1f}%)")
print(f"   (Null Island is at 0,0 in the Gulf of Guinea — a common GPS default error)")

# Check 4: Negative speeds
if "speed" in bad_gps.columns:
    negative_speeds = (bad_gps["speed"] < 0).sum()
    print(f"4. Negative speed values: {negative_speeds} ({negative_speeds/len(bad_gps)*100:.1f}%)")

# Check 5: Impossible speeds (> 100 mph for delivery trucks)
if "speed" in bad_gps.columns:
    impossible_speeds = (bad_gps["speed"] > 100).sum()
    print(f"5. Impossible speeds (> 100 mph): {impossible_speeds} ({impossible_speeds/len(bad_gps)*100:.1f}%)")

total_gps_issues = lat_out_of_range + lon_out_of_range + null_island
if "speed" in bad_gps.columns:
    total_gps_issues += negative_speeds + impossible_speeds

print(f"\n" + "=" * 60)
print(f"TOTAL GPS ISSUES: {total_gps_issues} across {len(bad_gps)} records")
print(f"Issue rate: {total_gps_issues/len(bad_gps)*100:.1f}%")
print(f"GreenRoute DAG threshold: 20%. Would this pass? {'YES' if total_gps_issues/len(bad_gps) <= 0.20 else 'NO'}")

In [ ]:
# What happens if bad GPS data enters the analytics pipeline?
print("Impact Analysis: Bad GPS Data in Analytics")
print("=" * 60)

# Filter to just the clean records
clean_mask = (
    (bad_gps["lat"].between(24, 50)) &
    (bad_gps["lon"].between(-130, -65)) &
    ~((bad_gps["lat"] == 0) & (bad_gps["lon"] == 0))
)
if "speed" in bad_gps.columns:
    clean_mask = clean_mask & (bad_gps["speed"] >= 0) & (bad_gps["speed"] <= 100)

clean_gps = bad_gps[clean_mask]
dirty_gps = bad_gps[~clean_mask]

print(f"\nClean records: {len(clean_gps)} ({len(clean_gps)/len(bad_gps)*100:.1f}%)")
print(f"Dirty records: {len(dirty_gps)} ({len(dirty_gps)/len(bad_gps)*100:.1f}%)")

if "speed" in bad_gps.columns:
    print(f"\nAverage speed WITH bad data:    {bad_gps['speed'].mean():.1f} mph")
    print(f"Average speed WITHOUT bad data: {clean_gps['speed'].mean():.1f} mph")
    print(f"\nThe dirty data {'inflates' if bad_gps['speed'].mean() > clean_gps['speed'].mean() else 'deflates'} "
          f"the average by {abs(bad_gps['speed'].mean() - clean_gps['speed'].mean()):.1f} mph.")
    print(f"This kind of silent corruption is why quality gates matter.")

### 2.4 Quality Gates in Airflow

A **quality gate** is a task in a DAG that:

1. Runs quality checks against the data
2. **Passes** — downstream tasks execute normally
3. **Fails** (raises an exception) — downstream tasks are **skipped** or marked as `upstream_failed`

This is the **circuit breaker pattern** applied to data pipelines. It prevents bad data from propagating through the system.

In the HealthPulse DAG, the `validate_data` task is the quality gate:

```python
def validate_patient_data(**context):
    # ... run checks ...
    if error_rate > 0.05:
        raise ValueError("Data quality check FAILED")
    # If we get here, the check passed
```

Because `transform_data` and `quality_report` depend on `validate_data`, they will **not run** if validation fails. The `load_to_warehouse` and `send_notification` tasks are also skipped — bad data never reaches the warehouse.

**This is a fundamental principle:** It is better for a pipeline to fail loudly than to succeed silently with bad data.

---

## Part 3: HealthPulse Complete Pipeline

Now let's work through the complete HealthPulse pipeline end-to-end, including triggering runs, monitoring execution, simulating failures, and running backfills.

### 3.1 Review the DAG

Let's look at the full DAG definition one more time, now that you understand quality gates.

In [ ]:
%%bash
# Display the HealthPulse DAG with line numbers for reference
echo "=== healthpulse_daily_pipeline.py ==="
docker exec airflow-webserver cat -n /opt/airflow/dags/healthpulse_daily_pipeline.py

**Key observations:**

- **Line ~33-41**: `default_args` — every task gets 2 retries with 5-minute delays and a 30-minute timeout
- **Line ~48-84**: `validate_patient_data()` — the quality gate function that checks for nulls, negatives, and future dates
- **Line ~78-80**: The circuit breaker — `raise ValueError(...)` if error rate exceeds 5%
- **Line ~119-127**: The DAG definition with `schedule="0 2 * * *"` and `catchup=False`
- **Line ~185-188**: The dependency chain that creates the diamond pattern

Note `catchup=False` — this tells Airflow NOT to run all the missed DAG runs between the start_date and today. Without this, Airflow would try to backfill every day since April 28.

### 3.2 Trigger and Monitor

In [ ]:
%%bash
# Trigger the HealthPulse DAG with a specific execution date
docker exec airflow-webserver airflow dags trigger \
    --exec-date 2026-04-28 \
    healthpulse_daily_pipeline

echo ""
echo "Triggered for execution date 2026-04-28."
echo "Monitor progress at: http://localhost:8080/dags/healthpulse_daily_pipeline/grid"

In [ ]:
# Wait for the run to complete
import time
print("Waiting 30 seconds for the DAG run to execute...")
time.sleep(30)
print("Done. Let's check the results.")

In [ ]:
%%bash
# Check task states for the run
echo "=== DAG Runs ==="
docker exec airflow-webserver airflow dags list-runs \
    -d healthpulse_daily_pipeline \
    -o table

echo ""
echo "=== Task Instance States ==="
# List all task instances and their states
docker exec airflow-webserver airflow tasks list healthpulse_daily_pipeline
echo ""
echo "Check the Grid view in the Web UI to see the color-coded task states:"
echo "  Green = success, Red = failed, Orange = running, Pink = upstream_failed"

In [ ]:
%%bash
# View the logs for the validate_data task
echo "=== Validate Data Task Logs ==="
echo "(Showing the most recent run)"
echo ""

# Get the latest run_id
RUN_ID=$(docker exec airflow-webserver airflow dags list-runs \
    -d healthpulse_daily_pipeline \
    -o plain --no-header 2>/dev/null | head -1 | awk '{print $3}')

if [ -n "$RUN_ID" ]; then
    docker exec airflow-webserver airflow tasks logs \
        healthpulse_daily_pipeline \
        validate_data \
        "$RUN_ID" 2>/dev/null | tail -30
else
    echo "No runs found yet. Trigger the DAG first (cell above)."
fi

### 3.3 Simulate a Failure — The Circuit Breaker in Action

Now let's see what happens when the quality gate **fails**. We'll copy the intentionally bad data file as the day's patient feed and trigger the DAG.

In [ ]:
%%bash
# Copy the bad data file as today's patient feed
# The DAG expects files at /opt/airflow/data/daily-patient-feed/{ds}.csv
# Since ./data is mounted at /opt/airflow/data in the container, we copy on the host

cp week11-pipeline-orchestration-modern-data-stack/data/healthpulse/quality-test-bad-data.csv \
   week11-pipeline-orchestration-modern-data-stack/data/healthpulse/daily-patient-feed/2026-05-01.csv

echo "Copied bad data as 2026-05-01 patient feed."
echo "This file has intentionally high error rates that will trip the quality gate."

In [ ]:
%%bash
# Trigger the DAG for the date with bad data
docker exec airflow-webserver airflow dags trigger \
    --exec-date 2026-05-01 \
    healthpulse_daily_pipeline

echo ""
echo "Triggered for 2026-05-01 (the bad data date)."
echo "Watch the Web UI — the validate_data task should turn RED."

In [ ]:
# Wait for the run to execute (including retries)
import time
print("Waiting 45 seconds for the run (including potential retries)...")
time.sleep(45)
print("Done. Let's check what happened.")

In [ ]:
%%bash
# Check the status — we expect validate_data to have failed
echo "=== DAG Run Status ==="
docker exec airflow-webserver airflow dags list-runs \
    -d healthpulse_daily_pipeline \
    -o table

echo ""
echo "Expected behavior:"
echo "  - ingest_patient_feed: SUCCESS (runs before quality check)"
echo "  - validate_data: FAILED (bad data exceeded 5% error threshold)"
echo "  - transform_data: UPSTREAM_FAILED (skipped because validation failed)"
echo "  - quality_report: UPSTREAM_FAILED (skipped because validation failed)"
echo "  - load_to_warehouse: UPSTREAM_FAILED (skipped)"
echo "  - send_notification: UPSTREAM_FAILED (skipped)"
echo ""
echo "This is the CIRCUIT BREAKER pattern: bad data was stopped at the gate."
echo "No corrupted data reached the warehouse."

### 3.4 Backfill

**Backfill** is the process of running a DAG for past dates that were missed or need reprocessing. Common scenarios:

- A pipeline was broken for 3 days and you need to catch up
- Upstream data was corrected and you need to reprocess
- A new column was added and historical data needs to be updated

Airflow has a built-in `backfill` command that creates a DAG run for each interval in a date range.

In [ ]:
%%bash
# Run a backfill for April 28-30
# This creates one DAG run per day (since the schedule is daily)
echo "Starting backfill for 2026-04-28 to 2026-04-30..."
echo "This will create 3 DAG runs — one for each day."
echo ""

docker exec airflow-scheduler airflow dags backfill \
    healthpulse_daily_pipeline \
    --start-date 2026-04-28 \
    --end-date 2026-04-30 \
    --reset-dagruns \
    2>&1 | tail -20

echo ""
echo "Backfill initiated. Check the Grid view to see multiple runs."

In [ ]:
%%bash
# After the backfill, check all runs
echo "=== All HealthPulse DAG Runs ==="
docker exec airflow-webserver airflow dags list-runs \
    -d healthpulse_daily_pipeline \
    -o table

echo ""
echo "Each row represents one day's pipeline execution."
echo "The backfill created runs for 2026-04-28, 2026-04-29, and 2026-04-30."

### 3.5 Idempotency Check

A pipeline is **idempotent** if running it multiple times with the same input produces the same output. This is critical for backfill operations — if you reprocess April 28, you should get the same result regardless of how many times you run it.

**How to achieve idempotency:**

- **Overwrite** output partitions instead of appending
- **Use execution date** (`{{ ds }}`) to determine which data to process
- **DELETE before INSERT** if writing to a database
- **Write to date-partitioned paths** (e.g., `/data/output/dt=2026-04-28/`)

In [ ]:
%%bash
# Trigger the same date twice to demonstrate idempotency
echo "=== First run for 2026-04-28 ==="
docker exec airflow-webserver airflow dags trigger \
    --exec-date 2026-04-28 \
    healthpulse_daily_pipeline 2>&1 | tail -3

echo ""
echo "=== Second run for 2026-04-28 ==="
docker exec airflow-webserver airflow dags trigger \
    --exec-date 2026-04-28 \
    healthpulse_daily_pipeline 2>&1 | tail -3

echo ""
echo "In a properly idempotent pipeline:"
echo "  - Both runs process the same input (2026-04-28.csv)"
echo "  - Both runs write to the same output partition (dt=2026-04-28)"
echo "  - The second run OVERWRITES the first run's output"
echo "  - The final state is identical regardless of how many times we run"
echo ""
echo "This is why the DAG uses {{ ds }} (execution date) everywhere —"
echo "it makes each run deterministic and repeatable."

---

## Part 4: GreenRoute Complete Pipeline

The GreenRoute pipeline orchestrates weekly fleet analytics. Unlike HealthPulse (daily patient data), GreenRoute processes **weekly GPS summaries** through a Spark-based aggregation pipeline.

### 4.1 Review the DAG

In [ ]:
%%bash
# Display the GreenRoute DAG with line numbers
echo "=== greenroute_weekly_report.py ==="
docker exec airflow-webserver cat -n /opt/airflow/dags/greenroute_weekly_report.py

**Key differences from HealthPulse:**

| Aspect | HealthPulse | GreenRoute |
|---|---|---|
| Schedule | Daily at 2 AM | Weekly (Mondays at 2 AM) |
| Dependency pattern | Diamond (parallel branches) | Linear chain |
| Quality checks | Null IDs, negative costs, future dates | GPS coordinates, Null Island, speed |
| Compute engine | Python (Airflow PythonOperator) | Spark (submitted via BashOperator) |
| Error threshold | 5% | 20% (GPS data is inherently noisier) |
| Retries | 2 | 3 (Spark jobs can fail due to resource contention) |

**Pipeline structure (linear chain):**

```
validate_gps_data → aggregate_fleet_metrics → generate_report → send_notification
```

The linear chain is appropriate here because each step depends on the previous one — you cannot aggregate before validating, and you cannot report before aggregating.

### 4.2 Trigger and Monitor

In [ ]:
%%bash
# Unpause and trigger the GreenRoute DAG
docker exec airflow-webserver airflow dags unpause greenroute_weekly_report
echo ""

docker exec airflow-webserver airflow dags trigger \
    --exec-date 2026-04-27 \
    greenroute_weekly_report

echo ""
echo "GreenRoute DAG triggered for week of 2026-04-27."
echo "Monitor at: http://localhost:8080/dags/greenroute_weekly_report/grid"

In [ ]:
import time
print("Waiting 30 seconds for the GreenRoute DAG to execute...")
time.sleep(30)
print("Done.")

In [ ]:
%%bash
# Check GreenRoute run status
echo "=== GreenRoute DAG Runs ==="
docker exec airflow-webserver airflow dags list-runs \
    -d greenroute_weekly_report \
    -o table

echo ""
echo "=== Task List ==="
docker exec airflow-webserver airflow tasks list greenroute_weekly_report

In [ ]:
%%bash
# View the GPS validation task logs
echo "=== GPS Validation Task Logs ==="

RUN_ID=$(docker exec airflow-webserver airflow dags list-runs \
    -d greenroute_weekly_report \
    -o plain --no-header 2>/dev/null | head -1 | awk '{print $3}')

if [ -n "$RUN_ID" ]; then
    docker exec airflow-webserver airflow tasks logs \
        greenroute_weekly_report \
        validate_gps_data \
        "$RUN_ID" 2>/dev/null | tail -20
else
    echo "No runs found. Trigger the DAG first."
fi

### 4.3 Quality Gate for GPS

Let's verify how the GPS quality gate works by looking at the validation thresholds and simulating a failure.

In [ ]:
# Demonstrate GPS quality checks matching what the Airflow DAG does
import random

print("Simulating the GPS quality check from the GreenRoute DAG")
print("=" * 60)

# Reproduce the logic from validate_gps_quality() in the DAG
random.seed(2026)
total_records = 1000
impossible_coords = 0
null_island = 0
negative_speeds = 0

for _ in range(total_records):
    lat = random.uniform(-95, 95)
    lon = random.uniform(-185, 185)
    speed = random.uniform(-5, 80)
    
    if abs(lat) > 90 or abs(lon) > 180:
        impossible_coords += 1
    if lat == 0.0 and lon == 0.0:
        null_island += 1
    if speed < 0:
        negative_speeds += 1

total_issues = impossible_coords + null_island + negative_speeds
issue_rate = total_issues / total_records

print(f"  Total records:       {total_records}")
print(f"  Impossible coords:   {impossible_coords}")
print(f"  Null Island (0,0):   {null_island}")
print(f"  Negative speeds:     {negative_speeds}")
print(f"  Issue rate:          {issue_rate:.2%}")
print(f"  Threshold:           20%")
print(f"  Result:              {'FAIL' if issue_rate > 0.20 else 'PASS'}")
print()
print("This matches exactly what the Airflow task computes.")
print("If the issue rate exceeds 20%, the task raises ValueError")
print("and all downstream tasks (aggregate, report, notify) are skipped.")

### 4.4 Cloud Migration: From Docker to AWS

Everything we have built in Weeks 8-11 runs on Docker containers on your laptop. In production, each component maps to an **AWS managed service**:

| On-Prem (Docker) | AWS Managed Service | Key Difference |
|---|---|---|
| **HDFS** (Week 8) | **Amazon S3** | No cluster to manage; pay per GB stored; 11 nines durability |
| **Spark** (Week 9) | **Amazon EMR** or **AWS Glue** | Auto-scaling clusters; pay per compute-hour; spot instances |
| **Kafka** (Week 10) | **Amazon MSK** | Managed brokers; auto-patching; integrated with IAM |
| **Airflow** (Week 11) | **Amazon MWAA** | Managed Airflow; auto-scaling workers; integrated with S3/EMR |
| **PostgreSQL** | **Amazon RDS** | Automated backups; multi-AZ; managed upgrades |

**The architecture stays the same — only the infrastructure changes.**

A production HealthPulse pipeline on AWS would look like:

```
MWAA (Airflow) schedules daily at 2 AM:
  1. S3 sensor waits for patient feed file in s3://healthpulse/raw/
  2. Glue job validates data quality
  3. EMR step transforms and deduplicates
  4. Glue crawler updates the Data Catalog
  5. SNS notification to the ops team
```

**Trade-offs of managed services:**

- **Pro:** No infrastructure management, automatic scaling, built-in monitoring
- **Pro:** Higher availability (multi-AZ) and durability (S3 11-nines)
- **Con:** Vendor lock-in — harder to switch cloud providers
- **Con:** Cost at scale can exceed self-managed infrastructure
- **Con:** Less control over versions and configuration

---

## Part 5: Reflection Questions

Take time to think about these questions. They connect the hands-on lab work to real-world data engineering decisions.

1. **Why is idempotency critical for backfill operations?**
   Consider what happens if a backfill runs twice for the same date — with an idempotent pipeline vs. a non-idempotent one (e.g., one that appends instead of overwrites).

2. **The HealthPulse quality gate checks for null patient_ids. What other quality checks would you add for healthcare data?**
   Consider HIPAA requirements, clinical accuracy, and downstream analytics. Think about: duplicate records, date consistency (discharge before admission?), valid diagnosis codes, and PHI exposure.

3. **GreenRoute's streaming pipeline (Week 10) runs 24/7. How would you use Airflow to monitor it without duplicating the streaming logic?**
   Hint: Airflow can run sensors and health checks without re-implementing the streaming processing.

4. **If you were building this pipeline on AWS, which managed service would replace each component? What trade-offs would you face?**
   Map each Docker container to its AWS equivalent and discuss cost, control, and operational complexity.

---

## Summary: The Complete Pipeline Journey

Over the past four weeks, we have built a complete data pipeline from scratch:

| Week | Layer | Technology | What We Built |
|---|---|---|---|
| **Week 8** | **Store** | HDFS, PostgreSQL | Distributed storage for raw and structured data |
| **Week 9** | **Transform** | Apache Spark | Batch transformations, aggregations, and joins |
| **Week 10** | **Stream** | Kafka, Spark Streaming | Real-time event processing and anomaly detection |
| **Week 11** | **Orchestrate** | Apache Airflow | Scheduled execution, quality gates, monitoring, backfill |

**What Airflow added to the stack:**

- **Scheduling** — Pipelines run automatically on cron schedules
- **Dependencies** — Tasks execute in the correct order with parallel branches
- **Quality Gates** — Bad data is caught before it enters the warehouse
- **Observability** — Every run is logged, every task state is tracked
- **Backfill** — Historical data can be reprocessed reliably
- **Idempotency** — Reprocessing produces consistent results

Together, these four layers form a **production-grade data platform** that can handle both batch and streaming workloads with confidence.

---

### Cleanup

When you are done with the lab, shut down the Docker environment:

In [ ]:
%%bash
# Shut down the Week 11 cluster
cd week11-pipeline-orchestration-modern-data-stack
docker compose down
echo ""
echo "All containers stopped. To also remove stored data:"
echo "  docker compose down -v"